# Analysis 06 temporal explainability

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# Analysis 06: Temporal Explainability

This notebook explains how dynamic features, lags, and sources affect the coastal transformer predictions.

Attention plots are included only as routing diagnostics. Integrated gradients and occlusion are the primary evidence of input sensitivity.

In [ ]:
from pathlib import Path

RESULTS_DIR = "results/FINAL_RESULTS_V2/18_cnn5chan_v1"
TRAINING_CONFIG_PATH = None
POINT_CENTRIC_DIR = None
STATIC_FEATURES_CSV = "data/processed/static/master_static_features.csv"
SPLIT = "val"  # train | val | test | all
DEVICE = "cuda"
RANDOM_SEED = 42

MAX_EXPLAIN_SAMPLES = 256
BACKGROUND_SAMPLES = 128
BATCH_SIZE = 8
TEMPORAL_IG_STEPS = 8
ATTENTION_MAX_SAMPLES = 16
ATTENTION_BATCH_SIZE = 4

TARGETS_TO_ANALYZE = ["hs", "tp", "dir", "dp"]
EXPLAIN_QUANTITY = "physical_prediction"  # physical_prediction | absolute_error | signed_error | entropy | selected_logit

USE_STRATIFIED_SAMPLING = True
SAMPLING_STRATA = ["site", "target_hs_bin", "target_tp_bin"]

DYNAMIC_GROUP_OVERRIDES = {}

In [ ]:
import importlib

import src.diagnostics.explainability as _explainability
import src.diagnostics as _diagnostics

importlib.reload(_explainability)
importlib.reload(_diagnostics)

from src.diagnostics import run_temporal_explainability_analysis

print("Results directory:", Path(RESULTS_DIR).resolve())
print("Split:", SPLIT)
print("Targets:", TARGETS_TO_ANALYZE)
print("Explain quantity:", EXPLAIN_QUANTITY)

In [ ]:
outputs = run_temporal_explainability_analysis(
    results_dir=RESULTS_DIR,
    split=SPLIT,
    device=DEVICE,
    max_explain_samples=MAX_EXPLAIN_SAMPLES,
    random_seed=RANDOM_SEED,
    targets_to_analyze=TARGETS_TO_ANALYZE,
    explain_quantity=EXPLAIN_QUANTITY,
    dynamic_group_overrides=DYNAMIC_GROUP_OVERRIDES,
    explain_batch_size=BATCH_SIZE,
    temporal_ig_steps=TEMPORAL_IG_STEPS,
    attention_max_samples=ATTENTION_MAX_SAMPLES,
    attention_batch_size=ATTENTION_BATCH_SIZE,
)
outputs["run_summary"]

## Cross-Attention Sanity Audit

This section checks whether the cross-attention plots are being extracted and interpreted correctly before averaging. Attention is routing evidence rather than causal feature importance, so any pattern here should be compared against integrated gradients and temporal occlusion before being treated as meaningful input importance.


In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.diagnostics.explainability import (
    circular_error_deg,
    extract_cross_attention_maps,
    load_model_for_explainability,
    load_prediction_frame,
    reconstruct_context_token_index_map,
    sample_explainability_subset,
)

ATTENTION_AUDIT_MAX_SAMPLES = min(ATTENTION_MAX_SAMPLES, MAX_EXPLAIN_SAMPLES)
ATTENTION_AUDIT_BATCH_SIZE = ATTENTION_BATCH_SIZE
INDIVIDUAL_SAMPLE_PLOTS = 6
HIGH_HS_QUANTILE = 0.75
LOW_HS_QUANTILE = 0.25
HIGH_ERROR_QUANTILE = 0.75
ATTENTION_ROW_SUM_ATOL = 1e-4
SAVE_AUDIT_FIGURES = True


def _to_numpy(value):
    if value is None:
        return None
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def warn_and_print(message):
    warnings.warn(message)
    print(f"WARNING: {message}")


def describe_attention_tensor(name, tensor, dimensions):
    if tensor is None:
        print(f"{name}: not returned by the model")
        return
    print(f"{name} shape: {tuple(tensor.shape)}")
    print(f"{name} dimensions: {', '.join(dimensions)}")


def print_row_sum_stats(name, row_sums):
    flat = np.asarray(row_sums, dtype=float).reshape(-1)
    print(f"{name}: min={flat.min():.6f}, max={flat.max():.6f}, mean={flat.mean():.6f}")


def prepare_attention_audit_frame(frame):
    out = frame.copy()
    if "target_dir_deg" not in out.columns and {"target_dir_sin", "target_dir_cos"}.issubset(
        out.columns
    ):
        out["target_dir_deg"] = (
            np.degrees(np.arctan2(out["target_dir_sin"], out["target_dir_cos"])) + 360.0
        ) % 360.0
    if "pred_dir_deg" not in out.columns and {"pred_dir_sin", "pred_dir_cos"}.issubset(out.columns):
        out["pred_dir_deg"] = (
            np.degrees(np.arctan2(out["pred_dir_sin"], out["pred_dir_cos"])) + 360.0
        ) % 360.0
    if "target_dp_deg" not in out.columns and {"target_dp_sin", "target_dp_cos"}.issubset(
        out.columns
    ):
        out["target_dp_deg"] = (
            np.degrees(np.arctan2(out["target_dp_sin"], out["target_dp_cos"])) + 360.0
        ) % 360.0
    if "pred_dp_deg" not in out.columns and {"pred_dp_sin", "pred_dp_cos"}.issubset(out.columns):
        out["pred_dp_deg"] = (
            np.degrees(np.arctan2(out["pred_dp_sin"], out["pred_dp_cos"])) + 360.0
        ) % 360.0

    out["hs_abs_error"] = np.abs(
        pd.to_numeric(out.get("pred_hs"), errors="coerce")
        - pd.to_numeric(out.get("target_hs"), errors="coerce")
    )
    out["tp_abs_error"] = np.abs(
        pd.to_numeric(out.get("pred_tp"), errors="coerce")
        - pd.to_numeric(out.get("target_tp"), errors="coerce")
    )
    if {"target_dir_deg", "pred_dir_deg"}.issubset(out.columns):
        out["dir_abs_error"] = np.abs(
            circular_error_deg(out["target_dir_deg"].to_numpy(), out["pred_dir_deg"].to_numpy())
        )
    if {"target_dp_deg", "pred_dp_deg"}.issubset(out.columns):
        out["dp_abs_error"] = np.abs(
            circular_error_deg(out["target_dp_deg"].to_numpy(), out["pred_dp_deg"].to_numpy())
        )
    return out


def select_quantile_indices(frame, column, *, quantile, side):
    series = pd.to_numeric(frame[column], errors="coerce").dropna()
    if series.empty:
        return []
    threshold = series.quantile(quantile)
    if side == "high":
        return series[series >= threshold].index.to_list()
    if side == "low":
        return series[series <= threshold].index.to_list()
    raise ValueError(f"Unsupported side: {side}")


def token_indices(token_map, selector):
    return token_map.loc[selector(token_map), "token_index"].astype(int).to_list()


def build_token_group_map(token_map):
    return {
        "final_timestep_token": token_indices(
            token_map, lambda df: (df["token_type"] == "dynamic timestep") & (df["lag"] == 0)
        ),
        "last_3_timesteps": token_indices(
            token_map,
            lambda df: (
                (df["token_type"] == "dynamic timestep") & (df["lag"] >= 0) & (df["lag"] < 3)
            ),
        ),
        "last_6_timesteps": token_indices(
            token_map,
            lambda df: (
                (df["token_type"] == "dynamic timestep") & (df["lag"] >= 0) & (df["lag"] < 6)
            ),
        ),
        "static_tokens": token_indices(token_map, lambda df: df["token_type"] == "static token"),
        "bathy_tokens": token_indices(token_map, lambda df: df["token_type"] == "bathy token"),
        "source_geometry_tokens": token_indices(
            token_map,
            lambda df: (
                df["token_type"].astype(str).str.contains("source geometry", case=False, na=False)
            ),
        ),
    }


def format_token_labels(token_map):
    labels = []
    for row in token_map.itertuples(index=False):
        if row.token_type == "dynamic timestep":
            labels.append(f"t-{int(row.lag)}")
        elif row.token_type == "static token":
            labels.append("static")
        elif row.token_type == "bathy token":
            labels.append(f"bathy_{int(row.token_subindex)}")
        elif row.token_type == "source token" and not pd.isna(row.source_index):
            labels.append(f"source_{int(row.source_index)}")
        elif row.token_type == "summary token":
            labels.append(f"summary_{int(row.token_subindex)}")
        else:
            labels.append(str(row.token_type))
    return labels


def token_boundaries(token_map):
    boundaries = []
    previous = None
    for row in token_map.itertuples(index=False):
        if previous is not None and row.token_type != previous:
            boundaries.append(row.token_index - 0.5)
        previous = row.token_type
    return boundaries


def resolve_batch_positions(frame_indices, index_to_batch_position):
    return [index_to_batch_position[idx] for idx in frame_indices if idx in index_to_batch_position]


def subset_attention_tokens(attention, token_map, selector):
    subset_map = token_map.loc[selector(token_map)].copy().reset_index(drop=True)
    if subset_map.empty:
        return attention[..., :0], subset_map, []
    token_idx = subset_map["token_index"].astype(int).to_numpy()
    subset_attention = attention[..., token_idx]
    subset_map["token_index"] = np.arange(len(subset_map), dtype=int)
    return subset_attention, subset_map, token_idx.tolist()


def plot_individual_sample_attention(
    attention,
    token_labels,
    boundaries,
    sample_labels,
    *,
    target_head,
    query_idx,
    max_samples,
    out_path=None,
):
    n = min(max_samples, attention.shape[0])
    ncols = min(3, max(1, n))
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 3.8 * nrows), squeeze=False)
    vmax = float(attention[:n, :, query_idx, :].max()) if n else 1.0
    tick_step = max(1, len(token_labels) // 16)
    flat_axes = axes.flatten()
    for panel_idx, ax in enumerate(flat_axes):
        if panel_idx >= n:
            ax.axis("off")
            continue
        arr = attention[panel_idx, :, query_idx, :]
        im = ax.imshow(arr, aspect="auto", cmap="viridis", vmin=0.0, vmax=vmax)
        for boundary in boundaries:
            ax.axvline(boundary, color="white", linewidth=0.8, alpha=0.6)
        ax.set_title(sample_labels[panel_idx])
        ax.set_ylabel("Attention head")
        ax.set_xlabel("Context token")
        ax.set_xticks(np.arange(0, len(token_labels), tick_step))
        ax.set_xticklabels(token_labels[::tick_step], rotation=45, ha="right")
        fig.colorbar(im, ax=ax, shrink=0.82)
    fig.suptitle(f"Cross-attention by individual sample: target head={target_head}")
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=180, bbox_inches="tight")
    return fig


def plot_attention_heads_separately(
    attention_by_head, token_labels, boundaries, *, target_head, cohort_name, out_path=None
):
    n_heads = attention_by_head.shape[0]
    ncols = 2
    nrows = int(math.ceil(n_heads / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(7.2 * ncols, 2.8 * nrows), squeeze=False, sharex=True
    )
    tick_step = max(1, len(token_labels) // 16)
    x = np.arange(len(token_labels))
    for head_idx, ax in enumerate(axes.flatten()):
        if head_idx >= n_heads:
            ax.axis("off")
            continue
        ax.plot(x, attention_by_head[head_idx], color="#1f77b4", linewidth=1.8)
        ax.set_title(f"Attention head {head_idx}")
        ax.set_ylabel("Weight")
        for boundary in boundaries:
            ax.axvline(boundary, color="black", linewidth=0.8, alpha=0.25)
        ax.set_xticks(np.arange(0, len(token_labels), tick_step))
        ax.set_xticklabels(token_labels[::tick_step], rotation=45, ha="right")
        ax.set_ylim(0.0, max(1e-6, float(attention_by_head.max()) * 1.05))
    fig.suptitle(f"Averaged attention for {target_head} by attention head ({cohort_name})")
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=180, bbox_inches="tight")
    return fig


def plot_cohort_attention_heatmaps(
    cohort_attention, token_labels, boundaries, *, target_head, out_path=None
):
    cohort_items = list(cohort_attention.items())
    fig, axes = plt.subplots(
        1, len(cohort_items), figsize=(6.2 * len(cohort_items), 4.6), squeeze=False, sharey=True
    )
    vmax = max(float(arr.max()) for _, arr in cohort_items if arr.size)
    tick_step = max(1, len(token_labels) // 16)
    for ax, (cohort_name, arr) in zip(axes.flatten(), cohort_items):
        im = ax.imshow(arr, aspect="auto", cmap="viridis", vmin=0.0, vmax=vmax)
        for boundary in boundaries:
            ax.axvline(boundary, color="white", linewidth=0.8, alpha=0.6)
        ax.set_title(cohort_name)
        ax.set_xlabel("Context token")
        ax.set_xticks(np.arange(0, len(token_labels), tick_step))
        ax.set_xticklabels(token_labels[::tick_step], rotation=45, ha="right")
        fig.colorbar(im, ax=ax, shrink=0.82)
    axes[0, 0].set_ylabel("Attention head")
    fig.suptitle(f"Averaged cross-attention comparison by cohort: target head={target_head}")
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=180, bbox_inches="tight")
    return fig

In [ ]:
runner = load_model_for_explainability(
    RESULTS_DIR,
    split=SPLIT,
    device=DEVICE,
    training_config_path=TRAINING_CONFIG_PATH,
    point_centric_dir=POINT_CENTRIC_DIR,
)
prediction_frame = load_prediction_frame(runner.bundle, split=SPLIT)
sampled = sample_explainability_subset(
    prediction_frame,
    max_samples=MAX_EXPLAIN_SAMPLES,
    random_seed=RANDOM_SEED,
    use_stratified_sampling=USE_STRATIFIED_SAMPLING,
    strata=SAMPLING_STRATA,
)
audit_frame = prepare_attention_audit_frame(sampled.iloc[:ATTENTION_AUDIT_MAX_SAMPLES].copy())
audit_sample_indices = audit_frame.index.to_list()
audit_dir = (
    runner.bundle.explainability_dir
    / "analysis_06_temporal_explainability"
    / "cross_attention_sanity_audit"
)
audit_dir.mkdir(parents=True, exist_ok=True)

attention_payload = extract_cross_attention_maps(
    runner,
    sample_indices=audit_sample_indices,
    batch_size=ATTENTION_AUDIT_BATCH_SIZE,
)
cross_attention = _to_numpy(attention_payload["cross_attention_weights"]).astype(
    np.float64, copy=False
)
source_attention = _to_numpy(attention_payload.get("source_attention_weights"))
task_order = list(attention_payload.get("task_order") or [])
if len(task_order) < cross_attention.shape[2]:
    task_order.extend([f"query_{idx}" for idx in range(len(task_order), cross_attention.shape[2])])

print(f"Attention audit sample count: {len(audit_sample_indices)}")
print(f"Task/query order: {task_order}")
describe_attention_tensor(
    "cross_attention_weights",
    cross_attention,
    ["batch", "attention heads", "query tokens / target heads", "context tokens"],
)
describe_attention_tensor(
    "source_attention_weights",
    source_attention,
    ["batch", "timesteps", "source indices"],
)

cross_row_sums = cross_attention.sum(axis=-1)
print_row_sum_stats("Cross-attention row sums over context tokens", cross_row_sums)
if not np.allclose(cross_row_sums, 1.0, atol=ATTENTION_ROW_SUM_ATOL, rtol=ATTENTION_ROW_SUM_ATOL):
    warn_and_print("Cross-attention row sums are not close to 1 over the context-token dimension.")

if source_attention is not None:
    source_row_sums = source_attention.sum(axis=-1)
    print_row_sum_stats("Source-attention row sums over source indices", source_row_sums)

token_info = reconstruct_context_token_index_map(
    runner,
    attention_payload,
    sample_indices=audit_sample_indices,
)
token_map = token_info["token_map"]
for message in token_info["warnings"]:
    warn_and_print(message)
if len(token_map) != token_info["expected_context_length"]:
    warn_and_print(
        f"Plotted token count ({len(token_map)}) does not match expected context length ({token_info['expected_context_length']})."
    )
if len(token_map) != cross_attention.shape[-1]:
    warn_and_print(
        f"Plotted token count ({len(token_map)}) does not match extracted context-token count ({cross_attention.shape[-1]})."
    )

display(token_map)
token_labels = format_token_labels(token_map)
boundaries = token_boundaries(token_map)
token_groups = build_token_group_map(token_map)
dynamic_attention, dynamic_token_map, dynamic_token_indices = subset_attention_tokens(
    cross_attention,
    token_map,
    lambda df: df["token_type"] == "dynamic timestep",
)
dynamic_token_labels = format_token_labels(dynamic_token_map)
dynamic_boundaries = token_boundaries(dynamic_token_map)
print(f"Dynamic-only token count: {len(dynamic_token_map)} timesteps")
if len(dynamic_token_map) != 48:
    warn_and_print(
        f"Expected 48 dynamic timestep tokens for the sequence-only plot, found {len(dynamic_token_map)}."
    )
display(dynamic_token_map)
if not token_groups["source_geometry_tokens"]:
    print(
        "Source geometry tokens are not present in the current cross-attention context; source geometry only affects upstream source aggregation."
    )

dominant_mass = cross_attention.max(axis=-1)
dominant_token = cross_attention.argmax(axis=-1)
dominant_rows = []
for query_idx, target_head in enumerate(task_order[: cross_attention.shape[2]]):
    flattened_mass = dominant_mass[:, :, query_idx].reshape(-1)
    flattened_tokens = dominant_token[:, :, query_idx].reshape(-1)
    mode_token = int(pd.Series(flattened_tokens).mode().iloc[0])
    dominant_rows.append(
        {
            "target_head": target_head,
            "mean_max_attention": float(flattened_mass.mean()),
            "max_attention": float(flattened_mass.max()),
            "most_common_argmax_token": mode_token,
            "most_common_argmax_label": token_labels[mode_token]
            if mode_token < len(token_labels)
            else f"token_{mode_token}",
        }
    )
    if np.all(dominant_mass[:, :, query_idx] > 0.90):
        warn_and_print(
            f"All sample/head rows for target head {target_head} place >90% attention on a single token."
        )
display(pd.DataFrame(dominant_rows))

index_to_batch_position = {idx: pos for pos, idx in enumerate(audit_frame.index)}
high_hs_indices = select_quantile_indices(
    audit_frame, "target_hs", quantile=HIGH_HS_QUANTILE, side="high"
)
low_hs_indices = select_quantile_indices(
    audit_frame, "target_hs", quantile=LOW_HS_QUANTILE, side="low"
)

mass_fraction_rows = []
sample_labels = [
    f"sample={idx} | {row.site} | {str(row.timestamp)[:16]}"
    for idx, row in audit_frame[["site", "timestamp"]].iterrows()
]
error_column_by_head = {
    "hs": "hs_abs_error",
    "tp": "tp_abs_error",
    "dir": "dir_abs_error",
    "dp": "dp_abs_error",
}

for query_idx, target_head in enumerate(task_order[: cross_attention.shape[2]]):
    target_attention = cross_attention[:, :, query_idx, :]
    error_column = error_column_by_head.get(target_head)
    high_error_indices = []
    if error_column in audit_frame.columns:
        high_error_indices = select_quantile_indices(
            audit_frame,
            error_column,
            quantile=HIGH_ERROR_QUANTILE,
            side="high",
        )

    cohort_positions = {
        "all samples": list(range(target_attention.shape[0])),
        "high-Hs samples": resolve_batch_positions(high_hs_indices, index_to_batch_position),
        "low-Hs samples": resolve_batch_positions(low_hs_indices, index_to_batch_position),
        "high-error samples": resolve_batch_positions(high_error_indices, index_to_batch_position),
    }

    individual_path = (
        audit_dir / f"individual_samples_{target_head}.png" if SAVE_AUDIT_FIGURES else None
    )
    plot_individual_sample_attention(
        cross_attention,
        token_labels,
        boundaries,
        sample_labels,
        target_head=target_head,
        query_idx=query_idx,
        max_samples=INDIVIDUAL_SAMPLE_PLOTS,
        out_path=individual_path,
    )
    plt.show()

    average_by_head = target_attention.mean(axis=0)
    separate_heads_path = (
        audit_dir / f"average_by_attention_head_{target_head}.png" if SAVE_AUDIT_FIGURES else None
    )
    plot_attention_heads_separately(
        average_by_head,
        token_labels,
        boundaries,
        target_head=target_head,
        cohort_name="all samples",
        out_path=separate_heads_path,
    )
    plt.show()

    dynamic_average_by_head = dynamic_attention[:, :, query_idx, :].mean(axis=0)
    dynamic_separate_heads_path = (
        audit_dir / f"average_by_attention_head_{target_head}_dynamic_only.png"
        if SAVE_AUDIT_FIGURES
        else None
    )
    plot_attention_heads_separately(
        dynamic_average_by_head,
        dynamic_token_labels,
        dynamic_boundaries,
        target_head=target_head,
        cohort_name="dynamic sequence only (static removed)",
        out_path=dynamic_separate_heads_path,
    )
    plt.show()

    cohort_means = {}
    for cohort_name, positions in cohort_positions.items():
        if not positions:
            warn_and_print(
                f"Cohort {cohort_name} is empty for target head {target_head}; skipping."
            )
            continue
        cohort_attention = target_attention[positions]
        cohort_means[cohort_name] = cohort_attention.mean(axis=0)
        for group_name, token_idx in token_groups.items():
            mass = (
                0.0
                if not token_idx
                else float(cohort_attention[:, :, token_idx].sum(axis=-1).mean())
            )
            mass_fraction_rows.append(
                {
                    "target_head": target_head,
                    "cohort": cohort_name,
                    "token_group": group_name,
                    "attention_mass_fraction": mass,
                }
            )

    comparison_path = (
        audit_dir / f"cohort_comparison_{target_head}.png" if SAVE_AUDIT_FIGURES else None
    )
    plot_cohort_attention_heatmaps(
        cohort_means,
        token_labels,
        boundaries,
        target_head=target_head,
        out_path=comparison_path,
    )
    plt.show()

    dynamic_cohort_means = {
        cohort_name: arr[:, dynamic_token_indices] for cohort_name, arr in cohort_means.items()
    }
    dynamic_comparison_path = (
        audit_dir / f"cohort_comparison_{target_head}_dynamic_only.png"
        if SAVE_AUDIT_FIGURES
        else None
    )
    plot_cohort_attention_heatmaps(
        dynamic_cohort_means,
        dynamic_token_labels,
        dynamic_boundaries,
        target_head=target_head,
        out_path=dynamic_comparison_path,
    )
    plt.show()

mass_fraction_df = pd.DataFrame(mass_fraction_rows)
if not mass_fraction_df.empty:
    mass_fraction_pivot = mass_fraction_df.pivot(
        index=["target_head", "cohort"], columns="token_group", values="attention_mass_fraction"
    ).reset_index()
    display(mass_fraction_pivot)
    mass_fraction_pivot.to_csv(audit_dir / "attention_mass_fractions.csv", index=False)

if SAVE_AUDIT_FIGURES:
    token_map.to_csv(audit_dir / "context_token_map.csv", index=False)
    pd.DataFrame(dominant_rows).to_csv(audit_dir / "dominant_token_summary.csv", index=False)

print(f"Saved attention audit artifacts under: {audit_dir}")